In [50]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.embeddings import FastEmbedEmbeddings
from rich import print
from IPython.display import Markdown, display
import os 
from dotenv import load_dotenv

In [51]:
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

In [52]:
# url = "https://365datascience.com/courses/"
url = "https://www.google.com/finance/beta"

In [53]:
loader = WebBaseLoader(url)

In [54]:
raw_docs = loader.load()

In [55]:
print(raw_docs)

[
    Document(
        metadata={
            'source': 'https://www.google.com/finance/beta',
            'title': 'Google Finance - Stock Market Prices, Real-time Quotes & Business News',
            'description': 'Google Finance provides real-time market quotes, international exchanges, up-to-date 
financial news, and analytics to help you make more informed trading and investment decisions.',
            'language': 'en-US'
        },
        page_content="Google Finance - Stock Market Prices, Real-time Quotes & Business NewsFinanceFinanceSign 
inFinanceFinanceFinancefinance_modeHomemanage_searchMarket trendsPortfoliosaddCreate portfolioWatchlistsaddCreate 
watchlistsettingsSettingsfeedbackSend feedbackexpand_allCompare marketsUSEuropeAsiaCurrenciesCryptoFuturesDow 
Jones49,230.71-0.16%-79.61Dow Jones49,230.710.16%S&P 5007,165.08+0.80%+56.68S&P 
5007,165.080.80%Nasdaq24,836.60+1.63%+398.09Nasdaq24,836.601.63%Russell2,787.00+0.43%+11.90Russell2,787.000.43%VIX1
8.71-3.11%-0.60VIX18.713.11%format_list_bulletedBuild a watchlistSign in to track investments you care aboutSign 
inYou may be interested ininfoThis list is generated from recent searches, followed securities, and other activity.
Learn moreAll data and information is provided “as is” for personal informational purposes only, and is not 
intended to be financial advice nor is it for trading purposes or investment, tax, legal, accounting or other 
advice. Google is not an investment adviser nor is it a financial adviser and expresses no view, recommendation or 
opinion with respect to any of the companies included in this list or any securities issued by those companies. 
Please consult your broker or financial representative to verify pricing before executing any trades. Learn 
moreAMZNAmazon.com Inc$263.99+$8.913.49%add_circle_outlineBABAAlibaba Group Holding Ltd - 
ADR$135.82+$4.123.13%add_circle_outlineIndexVIX18.71-0.603.11%add_circle_outlineMSFTMicrosoft 
Corp$424.60+$8.852.13%add_circle_outlineAAPLApple Inc$271.06-$2.370.87%add_circle_outlineFFord Motor 
Co$12.38-$0.1000.80%add_circle_outlineMarket trendsstacked_line_chartMarket indexesClimate 
leaderscopyrightCryptopaidCurrenciesToday's financial newsTop storiesLocal marketWorld marketsNews is 
loadingEarnings calendarBased on popular stocksApr28Bloom EnergyApr 28, 2026, 
4:00\u202fPMcalendar_add_onApr28RobinhoodApr 28, 2026, 4:00\u202fPMcalendar_add_onApr29Alphabet Inc.Apr 29, 2026, 
4:00\u202fPMcalendar_add_onApr29MicrosoftApr 29, 2026, 4:00\u202fPMcalendar_add_onApr29MetaApr 29, 2026, 
4:00\u202fPMcalendar_add_onApr29Amazon.comApr 29, 2026calendar_add_onMost followed on GoogleAAPLApple Inc3.71M 
following0.87%add_circle_outlineGOOGLAlphabet Inc Class A2.16M following1.63%add_circle_outlineMSFTMicrosoft 
Corp1.84M following2.13%add_circle_outlineAMZNAmazon.com Inc1.74M following3.49%add_circle_outlineMETAMeta 
Platforms Inc1.58M following2.41%add_circle_outlineTSLATesla Inc1.49M following0.69%add_circle_outlineMarket 
trendsMarket trends are loadingDiscover moreYou may be interested ininfoThis list is generated from recent 
searches, followed securities, and other activity. Learn moreAll data and information is provided “as is” for 
personal informational purposes only, and is not intended to be financial advice nor is it for trading purposes or 
investment, tax, legal, accounting or other advice. Google is not an investment adviser nor is it a financial 
adviser and expresses no view, recommendation or opinion with respect to any of the companies included in this list
or any securities issued by those companies. Please consult your broker or financial representative to verify 
pricing before executing any trades. Learn moreIndexDow Jones Industrial 
Average49,230.710.16%add_circle_outlineIndexS&P 5007,165.080.80%add_circle_outlineTSLATesla 
Inc$376.300.69%add_circle_outlineAAPLApple Inc$271.060.87%add_circle_outlineAMZNAmazon.com 
Inc$263.993.49%add_circle_outlineBABAAlibaba Group Holding Ltd - ADR$135.823.

In [56]:
text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(raw_docs)

In [57]:
embeddings = FastEmbedEmbeddings(model="BAAI/bge-base-en-v1.5")

Store data in a vectorstor

In [58]:
vectorstore = FAISS.from_documents(documents, embeddings)

In [59]:
retriever = vectorstore.as_retriever()

In [60]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    api_key= groq_key,
    model="openai/gpt-oss-120b",
    temperature=0
)

In [61]:
chat_hsitory = []
# query = "Which course on 365DataScience can help me learn AI?"
query = "What stock performed well today?"

In [62]:
relevant_docs = retriever.invoke(query)

In [63]:
context = "\n\n".join(doc.page_content for doc in relevant_docs)

In [64]:
display(Markdown(context))

Google Finance - Stock Market Prices, Real-time Quotes & Business NewsFinanceFinanceSign inFinanceFinanceFinancefinance_modeHomemanage_searchMarket trendsPortfoliosaddCreate portfolioWatchlistsaddCreate watchlistsettingsSettingsfeedbackSend feedbackexpand_allCompare marketsUSEuropeAsiaCurrenciesCryptoFuturesDow Jones49,230.71-0.16%-79.61Dow Jones49,230.710.16%S&P 5007,165.08+0.80%+56.68S&P 5007,165.080.80%Nasdaq24,836.60+1.63%+398.09Nasdaq24,836.601.63%Russell2,787.00+0.43%+11.90Russell2,787.000.43%VIX18.71-3.11%-0.60VIX18.713.11%format_list_bulletedBuild a watchlistSign in to track investments you care aboutSign inYou may be interested ininfoThis list is generated from recent searches, followed securities, and other activity. Learn moreAll data and information is provided “as is” for personal informational purposes only, and is not intended to be financial advice nor is it for trading purposes or investment, tax, legal, accounting or other advice. Google is not an investment adviser nor is it a financial adviser and expresses no view, recommendation or opinion with respect to any of the companies included in this list or any securities issued by those companies. Please consult your broker or financial representative to verify pricing before executing any trades. Learn moreAMZNAmazon.com Inc$263.99+$8.913.49%add_circle_outlineBABAAlibaba Group Holding Ltd - ADR$135.82+$4.123.13%add_circle_outlineIndexVIX18.71-0.603.11%add_circle_outlineMSFTMicrosoft Corp$424.60+$8.852.13%add_circle_outlineAAPLApple Inc$271.06-$2.370.87%add_circle_outlineFFord Motor Co$12.38-$0.1000.80%add_circle_outlineMarket trendsstacked_line_chartMarket indexesClimate leaderscopyrightCryptopaidCurrenciesToday's financial newsTop storiesLocal marketWorld marketsNews is loadingEarnings calendarBased on popular stocksApr28Bloom EnergyApr 28, 2026, 4:00 PMcalendar_add_onApr28RobinhoodApr 28, 2026, 4:00 PMcalendar_add_onApr29Alphabet Inc.Apr 29, 2026, 4:00 PMcalendar_add_onApr29MicrosoftApr 29, 2026, 4:00 PMcalendar_add_onApr29MetaApr 29, 2026, 4:00 PMcalendar_add_onApr29Amazon.comApr 29, 2026calendar_add_onMost followed on GoogleAAPLApple Inc3.71M following0.87%add_circle_outlineGOOGLAlphabet Inc Class A2.16M following1.63%add_circle_outlineMSFTMicrosoft Corp1.84M following2.13%add_circle_outlineAMZNAmazon.com Inc1.74M following3.49%add_circle_outlineMETAMeta Platforms Inc1.58M following2.41%add_circle_outlineTSLATesla Inc1.49M following0.69%add_circle_outlineMarket trendsMarket trends are loadingDiscover moreYou may be interested ininfoThis list is generated from recent searches, followed securities, and other activity. Learn moreAll data and information is provided “as is” for personal informational purposes only, and is not intended to be financial advice nor is it for trading purposes or investment, tax, legal, accounting or other advice. Google is not an investment adviser nor is it a financial adviser and expresses no view, recommendation or opinion with respect to any of the companies included in this list or any securities issued by those companies. Please consult your broker or financial representative to verify pricing before executing any trades. Learn moreIndexDow Jones Industrial Average49,230.710.16%add_circle_outlineIndexS&P 5007,165.080.80%add_circle_outlineTSLATesla Inc$376.300.69%add_circle_outlineAAPLApple Inc$271.060.87%add_circle_outlineAMZNAmazon.com Inc$263.993.49%add_circle_outlineBABAAlibaba Group Holding Ltd - ADR$135.823.13%add_circle_outlineMSFTMicrosoft Corp$424.602.13%add_circle_outlineNXSTNexstar Media Group Inc$203.780.51%add_circle_outlineIndexVIX18.713.11%add_circle_outlineFFord Motor Co$12.380.80%add_circle_outlineSPYState Street SPDR S&P 500 ETF Trust$713.940.77%add_circle_outlineIndexNasdaq Composite24,836.601.63%add_circle_outlineDISWalt Disney Co$102.601.01%add_circle_outlinePFEPfizer Inc$27.001.24%add_circle_outlineUALUnited Airlines Holdings Inc$93.001.92%add_circle_outlineNFLXNetflix

Disney Co$102.601.01%add_circle_outlinePFEPfizer Inc$27.001.24%add_circle_outlineUALUnited Airlines Holdings Inc$93.001.92%add_circle_outlineNFLXNetflix Inc$92.360.50%add_circle_outlineBABoeing Co$232.440.73%add_circle_outlineTAT&T Inc$26.201.54%add_circle_outlineHelpSend feedbackPrivacyTermsDisclaimerSearchClear searchClose searchGoogle appsMain menu

In [65]:
history_text = "\n".join(f"User: {q}\nAssistant: {a}" for a, q in chat_hsitory)

In [66]:
prompt = f""" 
Use the context below to answer the question

Conversation history:
{history_text}

context:
{context}

Question:
{query}
"""

In [67]:
response = llm.invoke(prompt)
chat_hsitory.append((query, response.content))

In [68]:
print(chat_hsitory)

[
    (
        'What stock performed well today?',
        'Based on the data, **Amazon.com Inc. (AMZN)** performed the best today – its price rose to\u202f$263.99, 
up **$8.91**, which is a **3.49\u202f% gain** for the session. (Alibaba (BABA) also did well, up 3.13\u202f%.)'
    )
]